<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex06-training-lab/Ex06_03_transfer_and_fine_tuning_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **complete** version: every cell is written out and runs as it stands. Read it, run it, and check what you see against the note under each section.*


<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.
- Goodfellow, Bengio & Courville, *Deep Learning*, MIT Press 2016.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_06 · Notebook 03 — A Second Machine, and Ten Labels Each

**Paired with L6.2 · Training Philosophies**

A classifier tells balanced from imbalance from bearing fault on machine A.
Machine B has a different accelerometer, gain and mounting, the same faults,
and **ten labelled samples per class**. You will

1. watch the machine-A model fail on machine B;
2. try three ways to use the same thirty labels: train from scratch, retrain
   only the last layer, fine-tune everything; and see what too large a
   learning rate does to the last of them;
3. ask how many labels you actually needed.

## 0 · Setup

**What the three cells below do.** The first fetches the library file
`Ex_6_core.py` when you run on Colab. The second keeps what this notebook saves
in your Google Drive, so the report notebook can read it later. The third
imports the libraries and fixes the random seed.

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_6_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex06-training-lab/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# outputs-cell v1 --------------------------------------------------------
# Later notebooks in this set read results that earlier ones save. On Google
# Colab every notebook runs on its own temporary machine, so a file saved
# here is not there when the next notebook opens. This cell keeps the
# results in your Google Drive instead: approve the access request when it
# appears. If you decline it, or have no Google Drive, the results are
# downloaded to your computer when saved and the notebook that needs them
# asks for them back. Locally this cell does nothing.
import Ex_6_core as core
core.keep_outputs()


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

import Ex_6_core as core

core.set_seed(0)
print("torch", torch.__version__, "| output dir:", core.OUTPUT_DIR)

## 1 · Two machines, same faults

**What transfer learning is.** Training a network needs many labelled
examples, and a label is expensive: someone has to open the machine, or wait for
it to fail, to know its true condition. **Transfer learning** takes a network
already trained on a task with plenty of labels and reuses it on a related task
with few. The first task is the **source**, the second the **target**.

**The situation here.** Machine A is the vibration problem of notebook 01: two
numbers per reading (the vibration at running speed, and in a high-frequency
band), three conditions (0 balanced, 1 imbalance, 2 bearing fault), and 360
labelled readings. Machine B is a new machine with the same three faults, but
its accelerometer has a different gain and is mounted elsewhere and at an
angle, so the same fault gives different numbers. Of its 180 readings, only ten
per condition are labelled. The question of this notebook: can the machine-A
network help on machine B, and how should it be used?

Look at them together before modelling anything. The classes are in the same
*arrangement* on both — that is the structure a transferred model can reuse —
but they sit somewhere else and at a different angle.

In [ ]:
# Two machines, the same three faults.
#   core.vibration_dataset()  machine A: plenty of labelled data
#   core.machine_b_dataset()  machine B: the new machine, shifted
#   core.few_shot_split(X, y, per_class, seed) -> a small labelled
#     set and the rest, which is the situation transfer is for
XA, yA = core.vibration_dataset()
XB, yB = core.machine_b_dataset()

fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.4), sharex=True, sharey=True)
core.plot_classes(XA, yA, ax=axes[0], title="machine A — 360 labelled samples")
core.plot_classes(XB, yB, ax=axes[1], title="machine B — 180, mostly unlabelled")
plt.show()

print("machine A feature means:", XA.mean(axis=0).round(3))
print("machine B feature means:", XB.mean(axis=0).round(3))

**What you should see.** Two clouds of three classes with the same internal
arrangement, machine B's shifted right, squashed on the vertical axis and
rotated. This is *covariate shift*: the relationship between fault and
vibration is unchanged, but the measurement of it is not.

## 2 · The machine-A model

**The goal.** Train the **source model**: a small network (two inputs, two
hidden layers of sixteen neurons, three outputs) on 240 labelled machine-A
readings, checked on the other 120. In a real project this is the model someone
else trained, or the one you trained last year. Then apply it to machine B
unchanged. Its accuracy there is the baseline every strategy in section 3 has
to beat.

**Two words for what follows.** The network's last layer, which turns the
features into the three class scores, is its **head**. Everything before it is
its **body**: the layers that turn raw readings into useful features. Transfer
learning keeps the body and treats the head as replaceable.

Train it properly on all of machine A, then leave it alone. It is the asset you
are trying to reuse.

### Your turn

In [ ]:
# the source model, on machine A --------------------------------------------
XA_tr, yA_tr = XA[:240], yA[:240]
XA_te, yA_te = XA[240:], yA[240:]

def fit(model, Xtr, ytr, lr=0.05, epochs=600, params=None):
    """Full-batch Adam on cross entropy; `params` limits what is trained."""
    optimiser = torch.optim.Adam(model.parameters() if params is None else params, lr=lr)
    for epoch in range(epochs):
        optimiser.zero_grad()
        loss = nn.CrossEntropyLoss()(model(torch.tensor(Xtr)), torch.tensor(ytr))
        loss.backward()
        optimiser.step()
    return model

def accuracy(model, X, y):
    with torch.no_grad():
        pred = model(torch.tensor(X)).argmax(dim=1).numpy()
    return float((pred == y).mean())

core.set_seed(0)
source = nn.Sequential(nn.Linear(2, 16), nn.Tanh(),
                       nn.Linear(16, 16), nn.Tanh(),
                       nn.Linear(16, 3))
fit(source, XA_tr, yA_tr)
acc_source_A = accuracy(source, XA_te, yA_te)
# ------------------------------------------------------------------------------

In [ ]:
# The source model, and what it scores on machine B untouched. That
# number is the baseline every later trick has to beat.
acc_source_B = accuracy(source, XB, yB)

print(f"machine A, held out : {acc_source_A:.3f}")
print(f"machine B, cold     : {acc_source_B:.3f}")
print(f"chance              : {1/3:.3f}")

fig, ax = plt.subplots(figsize=(6.4, 4.4))
with torch.no_grad():
    pred_B = source(torch.tensor(XB)).argmax(dim=1).numpy()
core.plot_classes(XB, yB, ax=ax, predictions=pred_B,
                  title="the machine-A model, applied to machine B")
plt.show()

**What you should see.** High accuracy on machine A, and something far worse on
machine B — poor, but usually still above the 0.333 you would get by guessing.

That residual skill is the whole premise of transfer. The model has not learned
nothing about machine B; it has learned the right *shape* of the problem and
the wrong *coordinates*. Section 3 tests whether it is cheaper to correct the
coordinates than to start again.

## 3 · Three ways to spend thirty labels

**The goal.** You have the machine-A model and thirty labelled machine-B
readings. There are three ways to use them, and this section tries all three on
exactly the same thirty readings, then tests each on the other 150 machine-B
readings, which none of them has seen.

- **Scratch** ignores machine A: a new network with random weights, trained on
  the thirty readings alone. It is the answer if transfer is not worth it.
- **Linear probe** keeps the machine-A body **frozen**, its weights not trained
  at all, and trains only a new head. There are few weights to fit, so thirty
  readings are enough, but the body cannot adapt to the new sensor.
- **Fine-tune** trains the whole machine-A network further on the thirty
  readings, at a **small learning rate**: 0.005, against 0.05 from scratch.
  Small steps adjust the body to machine B without erasing what it learned on
  machine A.

**What too large a learning rate does.** A fourth run fine-tunes at the
from-scratch learning rate, 0.05. Large steps can overwrite the pretrained body
in the first few updates, which is called **catastrophic forgetting**, and the
model then behaves much like one trained from scratch on thirty readings.
Compare it with the careful fine-tune.

Take ten labelled samples per class from machine B, and hold the rest back for
honest testing. Then three strategies on exactly the same thirty points.

| | what changes | why you might |
|---|---|---|
| **scratch** | a new network, random init | no dependence on machine A at all |
| **linear probe** | last layer only; body frozen | thirty points cannot support more |
| **fine-tune** | everything, small learning rate | the body is nearly right, not exactly |

### Your turn

In [ ]:
# scratch, linear probe, fine-tune ------------------------------------------
import copy
X_few, y_few, X_rest, y_rest = core.few_shot_split(XB, yB, per_class=10, seed=5)

core.set_seed(0)
scratch = nn.Sequential(nn.Linear(2, 16), nn.Tanh(),
                        nn.Linear(16, 16), nn.Tanh(),
                        nn.Linear(16, 3))
fit(scratch, X_few, y_few, lr=0.05)

probe = copy.deepcopy(source)
for p in probe.parameters():
    p.requires_grad = False
probe[-1] = nn.Linear(16, 3)
fit(probe, X_few, y_few, lr=0.05, params= probe[-1].parameters())

tuned = copy.deepcopy(source)
fit(tuned, X_few, y_few, lr= 0.005)

too_fast = copy.deepcopy(source)                  # the same, at the from-scratch learning rate
fit(too_fast, X_few, y_few, lr=0.05)

results = {"scratch":            accuracy(scratch,  X_rest, y_rest),
           "linear probe":       accuracy(probe,    X_rest, y_rest),
           "fine-tune":          accuracy(tuned,    X_rest, y_rest),
           "fine-tune, lr 0.05": accuracy(too_fast, X_rest, y_rest)}
# ------------------------------------------------------------------------------

In [ ]:
# Three ways to use thirty labels: train from scratch, fine-tune the
# whole source model, or freeze its features and retrain the head.
results["machine-A model, untouched"] = acc_source_B

print(core.error_table(
    [[name, f"{acc:.3f}"] for name, acc in results.items()],
    ["strategy", "accuracy on the 150 held-out machine-B samples"]))

fig, ax = plt.subplots(figsize=(7.4, 4.4))
names = list(results)
ax.barh(names, [results[n] for n in names],
        color=["#1f77b4", "#0f9d58", "#d94f2b", "#f4a300", "#888888"][:len(names)])
ax.axvline(1/3, color="#111111", lw=1.2, ls="--")
ax.text(1/3, -0.6, " chance", fontsize=8, color="#111111")
ax.set_xlim(0, 1); ax.set_xlabel("held-out accuracy")
ax.set_title("Thirty labels, five outcomes")
ax.grid(alpha=0.25, axis="x")
plt.show()

**What you should see.** Both transfer strategies beat training from scratch on
thirty points, and both beat leaving the machine-A model alone. Which of the two
wins is close on a body this small, so report what you measured, with the seed,
and not the ordering as a general law.

The fine-tune at learning rate 0.05 should give back much of the careful
fine-tune's advantage and land near the from-scratch result: that is
catastrophic forgetting. The rule of thumb it justifies, *fine-tune at roughly a
tenth of the learning rate you would use from scratch*, is the one L11 applies
to ResNet-18 without deriving it.

## 4 · How many labels did you actually need?

**The goal.** Transfer learning pays because labels are expensive. With enough
labels, a network trained from scratch should do as well, and the machine-A
model stops being worth the trouble. This section finds where that happens: it
repeats scratch and fine-tune with 2, 5, 10, 20 and 40 labelled readings per
condition, and plots held-out accuracy against the number of labels.

**How to read it.** Where the fine-tune curve sits well above scratch, transfer
is saving you labels. Where the two meet, it no longer is. Each budget is tested
on the machine-B readings left over, so the right-hand points rest on fewer
test readings and are noisier.

Thirty was asserted. Test it.

### Your turn

In [ ]:
# the label budget ----------------------------------------------------------
BUDGETS = [2, 5, 10, 20, 40]

curves = {"scratch": [], "fine-tune": []}
for n in BUDGETS:
    X_few_n, y_few_n, X_rest_n, y_rest_n = core.few_shot_split(XB, yB, per_class=n, seed=5)
    core.set_seed(0)
    m = nn.Sequential(nn.Linear(2, 16), nn.Tanh(),
                      nn.Linear(16, 16), nn.Tanh(),
                      nn.Linear(16, 3))
    fit(m, X_few_n, y_few_n, lr=0.05)
    curves["scratch"].append(accuracy(m, X_rest_n, y_rest_n))
    m = copy.deepcopy(source)
    fit(m, X_few_n, y_few_n, lr= 0.005)
    curves["fine-tune"].append(accuracy(m, X_rest_n, y_rest_n))
# ------------------------------------------------------------------------------

In [ ]:
# The same comparison at several label budgets. Where the lines cross
# is the answer to 'is transfer worth it here'.
fig, ax = plt.subplots(figsize=(7.0, 4.2))
for i, (name, curve) in enumerate(curves.items()):
    ax.plot(BUDGETS, curve, "o-", lw=1.9, ms=6,
            color=["#1f77b4", "#d94f2b"][i], label=name)
ax.axhline(acc_source_B, color="#888888", lw=1.4, ls=":",
           label="no adaptation")
ax.set_xscale("log"); ax.set_xticks(BUDGETS)
ax.set_xticklabels([str(b) for b in BUDGETS])
ax.set_xlabel("labelled samples per class"); ax.set_ylabel("held-out accuracy")
ax.set_title("Where the pretrained body stops paying for itself")
ax.legend(frameon=False, fontsize=9); ax.grid(alpha=0.25)
plt.show()

**What you should see.** The two curves are furthest apart on the left and
converge as labels accumulate. That is the general shape and the one worth
remembering: **transfer buys you labels.** Given enough of them, starting from
scratch catches up, and the pretrained model stops being worth the complexity.

The engineering question is never "is transfer better" but "how many labels can
I afford, and which side of the crossover does that put me on".

## 5 · Save

**What this does.** It writes the accuracies and the label-budget curves to a
file that the report in notebook 05 reads, and the fine-tuned model's weights
beside it. There is nothing to change here.

In [ ]:
# Saved for the report in notebook 05.
os.makedirs(core.OUTPUT_DIR, exist_ok=True)
path = os.path.join(core.OUTPUT_DIR, "nb03_transfer.npz")
np.savez(path,
         acc_source_A=acc_source_A, acc_source_B=acc_source_B,
         strategy_names=np.array(list(results)),
         strategy_acc=np.asarray(list(results.values()), dtype=float),
         budgets=np.asarray(BUDGETS),
         curve_scratch=np.asarray(curves["scratch"], dtype=float),
         curve_finetune=np.asarray(curves["fine-tune"], dtype=float))
torch.save(tuned.state_dict(), os.path.join(core.OUTPUT_DIR, "nb03_tuned.pt"))
print("wrote", path, "and nb03_tuned.pt")
core.saved(path, os.path.join(core.OUTPUT_DIR, "nb03_tuned.pt"))


## 6 · Before you move on

Answer these here. Each question builds part of an answer to one of the lecture's questions for the oral examination; the arrow under it says which, and the Questions slide at the end of the lecture has them in full.

1. The machine-A model scored above chance on machine B before any adaptation.
   What does that tell you about which layers transfer, and what would it have
   meant if it had scored *exactly* chance?
   *→ L6.2 Q1, Q2*
2. Section 3 fine-tuned the machine-A model twice: at a tenth of the
   from-scratch learning rate, and at the from-scratch rate itself. What does
   the larger rate do to the pretrained body, what is that failure called, and
   why does it make fine-tuning behave like training from scratch rather than
   like something worse?
   *→ L6.2 Q3*
3. You have thirty labels and a choice between a linear probe and full
   fine-tuning. What property of the *domain gap* should decide it — when should
   you unfreeze more than the head?
   *→ L6.2 Q3, Q4*
4. In L11 the pretrained body is ResNet-18 and the new data is a few hundred
   photographs. Which of the numbers in this notebook would you expect to change
   most, and in which direction? If you had thousands of unlabelled photographs
   and no labels at all, what task could self-supervised pre-training invent,
   and what would it buy you?
   *→ L6.2 Q4, Q5*


*Write your answers here. You will copy them into the report in notebook 05, which adds them to what you submit.*

1.
2.
3.
4.

---

Next: **[notebook 04](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex06-training-lab/Ex06_04_battery_arbitrage_light.ipynb)**, a battery that learns to trade.